# 將原始影像轉成ROI影像

In [1]:
import os
import cv2
import shutil

# 資料來源設定
txt_dir = r"C:\Users\howar\OneDrive\桌面\ALL_YOLO標記檔"
input_base = r"C:\temp\[Classification]normal_b_m_All_400"
output_base = r"C:\temp\[ROI_Classification]normal_b_m_All_400"

# 支援的影像格式
image_exts = ('.jpg', '.jpeg', '.png')

# 建立輸出資料夾
def ensure_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

# 讀取 YOLO 格式標註並回傳聯集 bounding box
def get_roi_from_yolo(txt_path, img_width, img_height):
    if not os.path.exists(txt_path):
        return None

    boxes = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            _, x_center, y_center, w, h = map(float, parts)
            x1 = int((x_center - w / 2) * img_width)
            y1 = int((y_center - h / 2) * img_height)
            x2 = int((x_center + w / 2) * img_width)
            y2 = int((y_center + h / 2) * img_height)
            boxes.append((x1, y1, x2, y2))
    if not boxes:
        return None
    # 計算所有 box 的聯集
    x1 = max(0, min([b[0] for b in boxes]))
    y1 = max(0, min([b[1] for b in boxes]))
    x2 = min(img_width, max([b[2] for b in boxes]))
    y2 = min(img_height, max([b[3] for b in boxes]))
    return (x1, y1, x2, y2)

# 主處理流程
subsets = ['train', 'val', 'test']
classes = ['benign', 'malignant']

for subset in subsets:
    for cls in classes:
        in_folder = os.path.join(input_base, subset, cls)
        out_folder = os.path.join(output_base, subset, cls)
        ensure_dir(out_folder)

        for fname in os.listdir(in_folder):
            if not fname.lower().endswith(image_exts):
                continue

            in_path = os.path.join(in_folder, fname)
            out_path = os.path.join(out_folder, f"[ROI]{fname}")
            name, _ = os.path.splitext(fname)
            txt_path = os.path.join(txt_dir, f"{name}.txt")

            # 讀圖
            img = cv2.imread(in_path)
            if img is None:
                print(f"[錯誤] 無法讀取圖片：{in_path}")
                continue

            h, w = img.shape[:2]
            roi = get_roi_from_yolo(txt_path, w, h)

            if roi:
                x1, y1, x2, y2 = roi
                cropped = img[y1:y2, x1:x2]
                cv2.imwrite(out_path, cropped)
            else:
                print(f"[警告] 無標註：{fname}，跳過裁剪")
                shutil.copyfile(in_path, out_path)  # 沒標註就原圖複製

print("✅ 所有影像已裁剪並儲存於 ROI 資料夾。")


✅ 所有影像已裁剪並儲存於 ROI 資料夾。
